# MongoDB Silver — EDA

## EDA Decision Report

### Purpose

This notebook performs exploratory data analysis on the MongoDB Bronze dataset before transformation into the Silver layer.

The purpose of the EDA is to understand the actual structure, data types, nested objects, arrays, missing values, distributions, identifiers, timestamps, and business-level data-quality conditions present in the source data.

The EDA does not modify the data and does not perform Silver transformations.

### EDA Principles

- Inspect the actual Bronze data before making transformation decisions.
- Preserve the original source grain during analysis.
- Do not delete or modify records during EDA.
- Do not treat intentional dirty data as a transformation problem.
- Distinguish technical data-shape issues from business-data-quality issues.
- Inspect nested structures and arrays explicitly.
- Treat MongoDB's internal `_id` as a source-system technical identifier, not as part of the analytical video schema.
- Use `video_id` as the analytical identifier for videos.
- Transformation decisions will be made only after the EDA is complete.

### Dataset Under Analysis

The MongoDB Bronze dataset represents video metadata.



### Scope

This notebook is limited to exploratory analysis of the MongoDB Bronze dataset.

No data transformation, cleaning, deletion, imputation, or business-rule correction is performed here.

### Expected Outcome

The EDA will provide the evidence required to create the MongoDB Silver Transformation Decision Report.

In [1]:
if 'spark' in globals():
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 17:13:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:

STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-01-Sep"
TABLE_NAME = "videos"

MONGO_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

In [4]:
df = spark.read.parquet(MONGO_BRONZE_PATH)

26/09/02 17:14:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Dataset Overview

In [5]:
print("Row count:")
print(df.count())

print("\nColumns:")
print(len(df.columns))

print("\nColumn names:")
print(df.columns)

Row count:


[Stage 1:>                                                          (0 + 1) / 1]

10000

Columns:
15

Column names:
['_id', 'competition', 'content_type', 'created_at', 'description', 'duration_sec', 'languages', 'last_updated', 'match_id', 'stats', 'status', 'tags', 'technical', 'title', 'video_id']


In [6]:
df.show(5, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------
 _id          | 6a97367f8bd7469c8b62b711                                                                                                        
 competition  | {Kings_Cup, Kings Cup, [Al_Riyadh, Hajer]}                                                                                      
 content_type | highlight                                                                                                                       
 created_at   | 2032-03-23 14:04:00                                                                                                             
 description  | Very carry take under agree. Girl phone between personal debate director dinner.                                                
 duration_sec | 6527                                                                                                              

Schema

In [7]:
print("MongoDB Bronze Schema:")
df.printSchema()

MongoDB Bronze Schema:
root
 |-- _id: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- teams: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- content_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- last_updated: timestamp (nullable = true)
 |-- match_id: string (nullable = true)
 |-- stats: struct (nullable = true)
 |    |-- views: integer (nullable = true)
 |    |-- likes: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical: struct (nullable = true)
 |    |-- max_resolution: string (nullable = true)
 |    |-- codec: string (nullabl

Data Types

In [8]:
print("Column data types:")

for field in df.schema.fields:
    print(f"{field.name}: {field.dataType}")

Column data types:
_id: StringType()
competition: StructType([StructField('id', StringType(), True), StructField('name', StringType(), True), StructField('teams', ArrayType(StringType(), True), True)])
content_type: StringType()
created_at: TimestampType()
description: StringType()
duration_sec: IntegerType()
languages: ArrayType(StringType(), True)
last_updated: TimestampType()
match_id: StringType()
stats: StructType([StructField('views', IntegerType(), True), StructField('likes', IntegerType(), True)])
status: StringType()
tags: ArrayType(StringType(), True)
technical: StructType([StructField('max_resolution', StringType(), True), StructField('codec', StringType(), True)])
title: StringType()
video_id: StringType()


Nested Structures

In [9]:
print("Competition:")
df.select("competition.*").printSchema()

print("\nTechnical:")
df.select("technical.*").printSchema()

print("\nStats:")
df.select("stats.*").printSchema()

Competition:
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- teams: array (nullable = true)
 |    |-- element: string (containsNull = true)


Technical:
root
 |-- max_resolution: string (nullable = true)
 |-- codec: string (nullable = true)


Stats:
root
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)



In [10]:
df.select(
    "competition.*",
    "technical.*",
    "stats.*"
).show(5, truncate=False, vertical=True)

-RECORD 0----------------------------------
 id             | Kings_Cup                
 name           | Kings Cup                
 teams          | [Al_Riyadh, Hajer]       
 max_resolution | 4k                       
 codec          | H.264                    
 views          | 42654                    
 likes          | 13413                    
-RECORD 1----------------------------------
 id             | Super_Cup                
 name           | Super Cup                
 teams          | [Al_Nassr, Al_Hilal]     
 max_resolution | 4k                       
 codec          | H.264                    
 views          | 396939                   
 likes          | 45652                    
-RECORD 2----------------------------------
 id             | RSL                      
 name           | RSL                      
 teams          | [Al_Qadsiah, Al_Khaleej] 
 max_resolution | 480p                     
 codec          | H.264                    
 views          | 233689        

Arrays

In [11]:
df.select(
    "video_id",
    F.size("languages").alias("languages_count"),
    F.size("tags").alias("tags_count"),
    F.size("competition.teams").alias("teams_count")
).show(10, truncate=False)

+------------------------------------+---------------+----------+-----------+
|video_id                            |languages_count|tags_count|teams_count|
+------------------------------------+---------------+----------+-----------+
|01a05ead-9bba-7c46-9da6-0cf37b66a413|1              |2         |2          |
|01a05ead-9bbb-712e-9c54-1d7f73dddd2e|1              |3         |2          |
|01a05ead-9bbc-74fc-aa0d-a799ddfda40c|2              |2         |2          |
|01a05ead-9bbd-7ccb-9c0e-1cb2b28e4899|2              |4         |2          |
|01a05ead-9bbe-7a52-8105-4b5c58abdb99|2              |4         |2          |
|01a05ead-9bbf-7b5c-936f-2a424cbee774|2              |2         |2          |
|01a05ead-9bc0-7daf-8328-fcccae350007|1              |3         |2          |
|01a05ead-9bc1-723f-8bd8-3d3011cd65a0|2              |4         |2          |
|01a05ead-9bc2-7c30-9f27-16a688e50ba5|2              |4         |2          |
|01a05ead-9bc3-7c26-92b6-fabe6e8c9ae4|2              |4         

In [12]:
print("Empty arrays:")

df.select(
    F.sum((F.size("languages") == 0).cast("int")).alias("empty_languages"),
    F.sum((F.size("tags") == 0).cast("int")).alias("empty_tags"),
    F.sum((F.size("competition.teams") == 0).cast("int")).alias("empty_teams"),
).show()

Empty arrays:
+---------------+----------+-----------+
|empty_languages|empty_tags|empty_teams|
+---------------+----------+-----------+
|              0|         0|          0|
+---------------+----------+-----------+



Missing / Null Values

In [13]:
print("Null counts:")

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

Null counts:
+---+-----------+------------+----------+-----------+------------+---------+------------+--------+-----+------+----+---------+-----+--------+
|_id|competition|content_type|created_at|description|duration_sec|languages|last_updated|match_id|stats|status|tags|technical|title|video_id|
+---+-----------+------------+----------+-----------+------------+---------+------------+--------+-----+------+----+---------+-----+--------+
|  0|          0|           0|         0|          0|        3254|        0|           0|       1|    0|     0|   0|        0|    1|       0|
+---+-----------+------------+----------+-----------+------------+---------+------------+--------+-----+------+----+---------+-----+--------+



In [14]:
df.select(
    F.sum(F.col("competition.id").isNull().cast("int")).alias("competition_id_nulls"),
    F.sum(F.col("competition.name").isNull().cast("int")).alias("competition_name_nulls"),
    F.sum(F.col("technical.max_resolution").isNull().cast("int")).alias("max_resolution_nulls"),
    F.sum(F.col("technical.codec").isNull().cast("int")).alias("codec_nulls"),
    F.sum(F.col("stats.views").isNull().cast("int")).alias("views_nulls"),
    F.sum(F.col("stats.likes").isNull().cast("int")).alias("likes_nulls"),
).show()

+--------------------+----------------------+--------------------+-----------+-----------+-----------+
|competition_id_nulls|competition_name_nulls|max_resolution_nulls|codec_nulls|views_nulls|likes_nulls|
+--------------------+----------------------+--------------------+-----------+-----------+-----------+
|                   0|                     0|                   0|          0|          0|          0|
+--------------------+----------------------+--------------------+-----------+-----------+-----------+



Optional / Sparse Fields

In [15]:
for c in df.columns:
    count = df.filter(F.col(c).isNull()).count()

    if count > 0:
        print(f"{c}: {count} nulls")

duration_sec: 3254 nulls
match_id: 1 nulls
title: 1 nulls


In [16]:
nested_columns = [
    "competition.id",
    "competition.name",
    "technical.max_resolution",
    "technical.codec",
    "stats.views",
    "stats.likes",
]

for c in nested_columns:
    count = df.filter(F.col(c).isNull()).count()
    print(f"{c}: {count} nulls")

competition.id: 0 nulls
competition.name: 0 nulls
technical.max_resolution: 0 nulls
technical.codec: 0 nulls
stats.views: 0 nulls
stats.likes: 0 nulls


Categorical Distributions

In [17]:
df.groupBy("content_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+------------+-----+
|content_type|count|
+------------+-----+
|replay      |3362 |
|live        |3332 |
|highlight   |3306 |
+------------+-----+



In [18]:
df.groupBy("status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+---------+-----+
|status   |count|
+---------+-----+
|archived |3466 |
|ended    |3280 |
|streaming|3254 |
+---------+-----+



In [19]:
df.groupBy("competition.name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+---------+-----+
|name     |count|
+---------+-----+
|Kings Cup|3379 |
|RSL      |3353 |
|Super Cup|3268 |
+---------+-----+



In [20]:
df.groupBy("technical.codec") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+-----+-----+
|codec|count|
+-----+-----+
|H.265|5029 |
|H.264|4971 |
+-----+-----+



In [21]:
df.groupBy("technical.max_resolution") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+--------------+-----+
|max_resolution|count|
+--------------+-----+
|1080p         |2539 |
|720p          |2504 |
|4k            |2496 |
|480p          |2461 |
+--------------+-----+



Uniqueness / Cardinality

In [22]:
total_rows = df.count()

distinct_video_ids = (
    df.select("video_id")
      .distinct()
      .count()
)

print("Total rows:", total_rows)
print("Distinct video_ids:", distinct_video_ids)
print("Duplicate video_ids:", total_rows - distinct_video_ids)

Total rows: 10000
Distinct video_ids: 10000
Duplicate video_ids: 0


In [23]:
print(
    "Null video_ids:",
    df.filter(F.col("video_id").isNull()).count()
)

Null video_ids: 0


Duration Distribution

In [24]:
df.select(
    F.min("duration_sec").alias("min_duration"),
    F.max("duration_sec").alias("max_duration"),
    F.avg("duration_sec").alias("avg_duration"),
    F.expr("percentile_approx(duration_sec, 0.5)").alias("median_duration"),
    F.expr("percentile_approx(duration_sec, 0.9)").alias("p90_duration"),
).show()

+------------+------------+------------------+---------------+------------+
|min_duration|max_duration|      avg_duration|median_duration|p90_duration|
+------------+------------+------------------+---------------+------------+
|           0|       20000|3751.2466646901867|           3758|        6520|
+------------+------------+------------------+---------------+------------+



Views / Likes Distribution

In [25]:
df.select(
    F.min("stats.views").alias("min_views"),
    F.max("stats.views").alias("max_views"),
    F.avg("stats.views").alias("avg_views"),
    F.expr("percentile_approx(stats.views, 0.5)").alias("median_views"),
).show()

+---------+---------+-----------+------------+
|min_views|max_views|  avg_views|median_views|
+---------+---------+-----------+------------+
|      100|   499950|249103.7077|      248158|
+---------+---------+-----------+------------+



In [26]:
df.select(
    F.min("stats.likes").alias("min_likes"),
    F.max("stats.likes").alias("max_likes"),
    F.avg("stats.likes").alias("avg_likes"),
    F.expr("percentile_approx(stats.likes, 0.5)").alias("median_likes"),
).show()

+---------+---------+----------+------------+
|min_likes|max_likes| avg_likes|median_likes|
+---------+---------+----------+------------+
|       10|    49999|24965.9846|       24789|
+---------+---------+----------+------------+



Timestamp Analysis

In [27]:
df.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.min("last_updated").alias("min_last_updated"),
    F.max("last_updated").alias("max_last_updated"),
).show()

+-------------------+-------------------+-------------------+-------------------+
|     min_created_at|     max_created_at|   min_last_updated|   max_last_updated|
+-------------------+-------------------+-------------------+-------------------+
|2032-03-22 16:58:00|2032-04-22 16:53:00|2032-03-22 16:58:00|2032-04-22 18:56:00|
+-------------------+-------------------+-------------------+-------------------+



In [28]:
created_after_updated = df.filter(
    F.col("last_updated") < F.col("created_at")
).count()

print("last_updated before created_at:", created_after_updated)

last_updated before created_at: 0


Business Validation

In [29]:
print(
    "Duration = 0:",
    df.filter(F.col("duration_sec") == 0).count()
)

Duration = 0: 1


In [30]:
print(
    "Duration > 10800:",
    df.filter(F.col("duration_sec") > 10800).count()
)

Duration > 10800: 1


In [31]:
print(
    "Live videos without match_id:",
    df.filter(
        (F.col("content_type") == "live")
        & F.col("match_id").isNull()
    ).count()
)

Live videos without match_id: 1


In [32]:
print(
    "Live content with ended status:",
    df.filter(
        (F.col("content_type") == "live")
        & (F.col("status") == "ended")
    ).count()
)

Live content with ended status: 1124


In [33]:
print(
    "Likes > views:",
    df.filter(
        F.col("stats.likes") > F.col("stats.views")
    ).count()
)

Likes > views: 516


In [34]:
competition_tag_matches = df.filter(
    F.array_contains(
        F.col("tags"),
        "Kings_Cup"
    )
).count()

print("Videos containing Kings_Cup tag:", competition_tag_matches)

Videos containing Kings_Cup tag: 3339


Arrays

In [35]:
print("Languages:")
df.select(F.explode("languages").alias("language")) \
    .groupBy("language") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

Languages:
+--------+-----+
|language|count|
+--------+-----+
|ar      |7497 |
|en      |7461 |
+--------+-----+



In [36]:
print("Tags:")
df.select(F.explode("tags").alias("tag")) \
    .groupBy("tag") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

Tags:
+----------+-----+
|tag       |count|
+----------+-----+
|goals     |3405 |
|VAR       |3401 |
|Kings_Cup |3339 |
|live      |3336 |
|highlights|3327 |
|RSL       |3307 |
|football  |3303 |
|replay    |3279 |
|Super_Cup |3278 |
+----------+-----+



Final EDA Summary

In [37]:
print("========================================")
print("MongoDB EDA Summary")
print("========================================")

print(f"Total videos: {df.count()}")

print(
    "Duplicate video_ids:",
    df.count() - df.select("video_id").distinct().count()
)

print(
    "Null video_ids:",
    df.filter(F.col("video_id").isNull()).count()
)

print(
    "Missing titles:",
    df.filter(F.col("title").isNull()).count()
)

print(
    "Zero durations:",
    df.filter(F.col("duration_sec") == 0).count()
)

print(
    "Durations > 10800:",
    df.filter(F.col("duration_sec") > 10800).count()
)

print(
    "Live without match_id:",
    df.filter(
        (F.col("content_type") == "live")
        & F.col("match_id").isNull()
    ).count()
)

print(
    "Likes > views:",
    df.filter(
        F.col("stats.likes") > F.col("stats.views")
    ).count()
)

print(
    "last_updated before created_at:",
    df.filter(
        F.col("last_updated") < F.col("created_at")
    ).count()
)

print("========================================")

MongoDB EDA Summary
Total videos: 10000
Duplicate video_ids: 0
Null video_ids: 0
Missing titles: 1
Zero durations: 1
Durations > 10800: 1
Live without match_id: 1
Likes > views: 516
last_updated before created_at: 0
